In [3]:
import pandas as pd
import numpy as np

In [4]:
data_ngfs = pd.read_excel(
    "data/ngfs_data/ngfs_macro_trimestriel.xlsx"
)

data_ngfs['variable'].unique()



array(['Central bank Intervention rate (policy interest rate)',
       'Consumption (private)', 'Domestic demand',
       'Effective exchange rate', 'Exchange rate',
       'Exports (goods and services)', 'GDP Growth Rate',
       'Gov. consumption', 'Imports (goods and services)',
       'Inflation rate', 'Investment (gov.)',
       'Investment (private sector)', 'Long term interest rate',
       'Long term real interest rate',
       'Trend output for capacity utilisation', 'Unemployment rate',
       'Carbon pricing', 'Coal price', 'Energy consumption (total)',
       'Equity prices', 'Gas price', 'Gross domestic income',
       'Gross operating surplus', 'House prices (residential)',
       'Oil price', 'Productivity (output per hour worked)',
       'Quarterly consumption of coal', 'Quarterly consumption of gas',
       'Quarterly consumption of non-carbon',
       'Quarterly consumption of oil', 'Real personal disposable income',
       'Revenue from CB or BCA tax',
       'Volum

In [5]:
[
'Central bank Intervention rate (policy interest rate)',
'Effective exchange rate',
'Equity prices',
'GDP Growth Rate',
'House prices (residential)',
'Inflation rate',
'Long term interest rate',
'Oil price',
'Unemployment rate'
]

['Central bank Intervention rate (policy interest rate)',
 'Effective exchange rate',
 'Equity prices',
 'GDP Growth Rate',
 'House prices (residential)',
 'Inflation rate',
 'Long term interest rate',
 'Oil price',
 'Unemployment rate']

In [6]:
NGFS_TO_MODEL = {
    "Central bank Intervention rate (policy interest rate)":
        "US_Central_bank_Intervention_rate_policy_interest_rate",

    "Effective exchange rate":
        "US_Effective_exchange_rate",

    "Equity prices":
        "US_Equity_prices",

    "GDP Growth Rate":
        "US_GDP_Growth_Rate",

    "House prices (residential)":
        "US_House_prices_residential",

    "Inflation rate":
        "US_Inflation_rate",

    "Long term interest rate":
        "US_Long_term_interest_rate",

    "Oil price":
        "US_Oil_price",

    "Unemployment rate":
        "US_Unemployment_rate",
}

In [8]:
data_ngfs["zone"] = data_ngfs["zone"].ffill()

In [9]:
data_ngfs.head()

,zone,variable,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,2023-Q3,2023-Q4,...,2047-Q4,2048-Q1,2048-Q2,2048-Q3,2048-Q4,2049-Q1,2049-Q2,2049-Q3,2049-Q4,2050-Q1
0,EU,Central bank Intervention rate (policy interes...,0.666667,1.594365,2.522064,3.449763,4.377462,4.494107,4.610752,4.727397,...,3.737243,3.737243,3.737243,3.737243,3.737243,3.737243,3.737243,3.737243,3.737243,3.737243
1,EU,Consumption (private),5967.662231,5975.562408,5983.462585,5991.362762,5999.262939,6006.761536,6014.260132,6021.758728,...,7510.032715,7522.438110,7534.423065,7546.408020,7558.392975,7570.377930,7581.977295,7593.576660,7605.176025,7616.775391
2,EU,Domestic demand,10909.079590,10922.391785,10935.703979,10949.016174,10962.328369,10984.926636,11007.524902,11030.123169,...,14444.852661,14482.053223,14519.342346,14556.631470,14593.920593,14631.209717,14668.538147,14705.866577,14743.195007,14780.523438
3,EU,Effective exchange rate,104.197612,105.617902,107.038192,108.458482,109.878772,110.332601,110.786430,111.240258,...,121.020298,121.133349,121.246840,121.360331,121.473822,121.587313,121.701236,121.815159,121.929082,122.043005
4,EU,Exchange rate,104.197612,103.476717,102.755823,102.034928,101.314034,101.137369,100.960704,100.784040,...,99.624373,99.624373,99.624373,99.624373,99.624373,99.624373,99.624373,99.624373,99.624373,99.624373


In [10]:
def build_X_from_ngfs_trimestriel(df_ngfs, zone, var_mapping):

    # 1️⃣ Filtrer zone US
    df = df_ngfs[df_ngfs["zone"] == zone].copy()

    # 2️⃣ Garder uniquement variables utiles
    df = df[df["variable"].isin(var_mapping.keys())].copy()

    # 3️⃣ Colonnes trimestrielles
    quarter_cols = [c for c in df.columns if "-Q" in str(c)]

    # 4️⃣ Wide → long
    df_long = df.melt(
        id_vars=["zone", "variable"],
        value_vars=quarter_cols,
        var_name="date",
        value_name="value"
    )

    # 5️⃣ Convertir date
    df_long["date"] = pd.PeriodIndex(df_long["date"], freq="Q").to_timestamp()

    # 6️⃣ Renommer variables selon ton modèle
    df_long["var_model"] = df_long["variable"].map(var_mapping)

    # 7️⃣ Pivot final : dates × variables
    X = df_long.pivot_table(
        index="date",
        columns="var_model",
        values="value",
        aggfunc="mean"
    ).sort_index()

    return X

In [11]:
macro_cols = [
    'US_Central_bank_Intervention_rate_policy_interest_rate',
    'US_Effective_exchange_rate',
    'US_Equity_prices',
    'US_GDP_Growth_Rate',
    'US_House_prices_residential',
    'US_Inflation_rate',
    'US_Long_term_interest_rate',
    'US_Oil_price',
    'US_Unemployment_rate'
]

In [12]:
X_scenario = build_X_from_ngfs_trimestriel(
    df_ngfs=data_ngfs,
    zone="US",
    var_mapping=NGFS_TO_MODEL
)

# Mettre dans le bon ordre
X_scenario = X_scenario[macro_cols]

X_scenario.head()

var_model,US_Central_bank_Intervention_rate_policy_interest_rate,US_Effective_exchange_rate,US_Equity_prices,US_GDP_Growth_Rate,US_House_prices_residential,US_Inflation_rate,US_Long_term_interest_rate,US_Oil_price,US_Unemployment_rate
date,,,,,,,,,
2022-01-01,1.895833,110.233435,157.738774,0.328440,136.802637,8.007718,2.951667,98.996328,3.641666
2022-04-01,2.748228,110.349450,158.250548,0.328440,138.785644,7.177069,3.203569,95.141495,3.638935
2022-07-01,3.600623,110.465465,158.762322,0.328440,140.768650,6.346419,3.455472,91.286663,3.636204
2022-10-01,4.453018,110.581481,159.274096,0.328440,142.751657,5.515769,3.707374,87.431830,3.633473
2023-01-01,5.305413,110.697496,159.785870,2.463064,144.734664,4.685120,3.959277,83.576998,3.630742


In [34]:
X_scenario.to_csv("data/ngfs_data/ngfs_us.csv")

In [1]:
import pandas as pd

file_path = "data/ngfs_data/ngfs_macro_trimestriel.xlsx"

xls = pd.ExcelFile(file_path)

print(xls.sheet_names)

['Baseline', 'Below 2°C', 'Current Policies', 'Delayed transition', 'Fragmented World', 'Nationally Determined Contribut', 'Net Zero 2050', 'Récapitulatif']


In [17]:
df_baseline = pd.read_excel(file_path, sheet_name='Below 2°C')

In [18]:
df_baseline

,zone,variable,2022-Q1,2022-Q2,2022-Q3,2022-Q4,2023-Q1,2023-Q2,2023-Q3,2023-Q4,...,2047-Q4,2048-Q1,2048-Q2,2048-Q3,2048-Q4,2049-Q1,2049-Q2,2049-Q3,2049-Q4,2050-Q1
0,EU,Central bank Intervention rate (policy interes...,0.666667,1.624615,2.582564,3.540512,4.498461,4.598145,4.697829,4.797513,...,4.327303,4.325384,4.178349,4.031314,3.884278,3.737243,3.737243,3.737243,3.737243,3.737243
1,NaN,Consumption (private),5967.662231,5954.341248,5941.020264,5927.699280,5914.378296,5915.197784,5916.017273,5916.836761,...,7109.613831,6984.584473,6988.957397,6993.330322,6997.703247,7002.076172,7011.868652,7021.661133,7031.453613,7041.246094
2,NaN,Domestic demand,10909.079590,10889.159363,10869.239136,10849.318909,10829.398682,10835.219971,10841.041260,10846.862549,...,13668.813965,13457.184814,13495.033752,13532.882690,13570.731628,13608.580566,13642.185425,13675.790283,13709.395142,13743.000000
3,NaN,Effective exchange rate,104.197612,105.645256,107.092901,108.540545,109.988189,110.486500,110.984810,111.483120,...,122.737990,122.848999,122.533577,122.218156,121.902734,121.587313,121.701236,121.815159,121.929082,122.043005
4,NaN,Exchange rate,104.197612,103.295241,102.392870,101.490499,100.588128,100.315083,100.042038,99.768993,...,95.452565,95.452404,96.495396,97.538388,98.581380,99.624373,99.624373,99.624373,99.624373,99.624373
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,NaN,Real personal disposable income,15627.978516,15719.208130,15810.437744,15901.667358,15992.896973,15965.431946,15937.966919,15910.501892,...,21617.226440,21244.638184,21467.961885,21691.285586,21914.609287,22137.932988,22204.381463,22270.829939,22337.278414,22403.726890
77,NaN,Revenue from CB or BCA tax,0.000000,21.682688,43.365376,65.048063,86.730751,94.990822,103.250894,111.510965,...,551.199524,541.257874,405.943405,270.628937,135.314468,0.000000,0.000000,0.000000,0.000000,0.000000
78,NaN,Trend output for capacity utilisation,88771.253906,88925.289551,89079.325195,89233.360840,89387.396484,89553.085938,89718.775391,89884.464844,...,125126.509766,123294.087891,123538.917969,123783.748047,124028.578125,124273.408203,124628.519531,124983.630859,125338.742188,125693.853516
79,NaN,Unemployment rate,3.641666,3.635445,3.629223,3.623001,3.616779,3.712219,3.807659,3.903099,...,4.523479,4.477424,4.493731,4.510039,4.526346,4.542653,4.545687,4.548721,4.551755,4.554788


In [13]:
df_baseline = build_X_from_ngfs_trimestriel(
    df_ngfs=data_ngfs,
    zone="US",
    var_mapping=NGFS_TO_MODEL
)

In [14]:
df_baseline

var_model,US_Central_bank_Intervention_rate_policy_interest_rate,US_Effective_exchange_rate,US_Equity_prices,US_GDP_Growth_Rate,US_House_prices_residential,US_Inflation_rate,US_Long_term_interest_rate,US_Oil_price,US_Unemployment_rate
date,,,,,,,,,
2022-01-01,1.895833,110.233435,157.738774,0.328440,136.802637,8.007718,2.951667,98.996328,3.641666
2022-04-01,2.748228,110.349450,158.250548,0.328440,138.785644,7.177069,3.203569,95.141495,3.638935
2022-07-01,3.600623,110.465465,158.762322,0.328440,140.768650,6.346419,3.455472,91.286663,3.636204
2022-10-01,4.453018,110.581481,159.274096,0.328440,142.751657,5.515769,3.707374,87.431830,3.633473
2023-01-01,5.305413,110.697496,159.785870,2.463064,144.734664,4.685120,3.959277,83.576998,3.630742
...,...,...,...,...,...,...,...,...,...
2049-01-01,3.318947,118.022898,387.540381,1.175574,247.022541,2.861358,3.248340,104.862288,4.715063
2049-04-01,3.318947,118.139290,391.142658,1.175574,248.438675,2.864759,3.248340,105.199467,4.715424
2049-07-01,3.318947,118.255683,394.744936,1.175574,249.854810,2.868160,3.248340,105.536646,4.715785


In [15]:
scenarios = [
    'Baseline',
    'Below 2°C',
    'Current Policies',
    'Delayed transition',
    'Fragmented World',
    'Nationally Determined Contribut',
    'Net Zero 2050'
]

In [19]:
import pandas as pd
import os

file_path = "data/ngfs_data/ngfs_macro_trimestriel.xlsx"

output_dir = "data/ngfs_data"
os.makedirs(output_dir, exist_ok=True)

for scen in scenarios:
    
    print(f"Traitement scénario : {scen}")
    
    # 1️⃣ Lire la feuille
    df_ngfs = pd.read_excel(file_path, sheet_name=scen)
    df_ngfs["zone"] = df_ngfs["zone"].ffill()
    
    # 2️⃣ Construire X pour US
    df_X = build_X_from_ngfs_trimestriel(
        df_ngfs=df_ngfs,
        zone="US",
        var_mapping=NGFS_TO_MODEL
    )
    
    # 3️⃣ Sauvegarder CSV propre
    filename = scen.replace(" ", "_").replace("°", "").replace("²", "2")
    csv_path = os.path.join(output_dir, f"X_{filename}.csv")
    
    df_X.to_csv(csv_path, index=True)
    
    print(f"✔ Sauvegardé : {csv_path}")

Traitement scénario : Baseline
✔ Sauvegardé : data/ngfs_data\X_Baseline.csv
Traitement scénario : Below 2°C
✔ Sauvegardé : data/ngfs_data\X_Below_2C.csv
Traitement scénario : Current Policies
✔ Sauvegardé : data/ngfs_data\X_Current_Policies.csv
Traitement scénario : Delayed transition
✔ Sauvegardé : data/ngfs_data\X_Delayed_transition.csv
Traitement scénario : Fragmented World
✔ Sauvegardé : data/ngfs_data\X_Fragmented_World.csv
Traitement scénario : Nationally Determined Contribut
✔ Sauvegardé : data/ngfs_data\X_Nationally_Determined_Contribut.csv
Traitement scénario : Net Zero 2050
✔ Sauvegardé : data/ngfs_data\X_Net_Zero_2050.csv
